<a href="https://colab.research.google.com/github/JuanCappelletti/7-Preprocesamiento/blob/master/01_Consolidar_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
# ================================================================
# ANALISIS COMPLETO DE CAUDAL - GOOGLE COLAB
#
# TODO EN UNA SOLA CELDA
#
# ACCIONES DEL USUARIO DESTACADAS:
# 1 - CARGAR ARCHIVOS
# 2 - ESCRIBIR NOMBRE DEL INFORME
#
# GENERA:
# - Excel
# - HTML interactivo
# ================================================================


# ================================================================
# INSTALAR LIBRERIAS
# ================================================================

!pip install -q plotly openpyxl


# ================================================================
# IMPORTAR LIBRERIAS
# ================================================================

import io
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from plotly.offline import plot
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from google.colab import files
from IPython.display import display, HTML


# ================================================================
# FUNCION PARA MOSTRAR PEDIDOS GRANDES AL USUARIO
# ================================================================

def cartel_usuario(titulo, mensaje):

    display(
        HTML(
            f"""
            <div style="
                background:#fff3cd;
                border:4px solid #ff9800;
                border-radius:16px;
                padding:25px;
                margin:25px 0;
                text-align:center;
                box-shadow:0 4px 12px rgba(0,0,0,0.15);
            ">

                <div style="
                    font-size:30px;
                    font-weight:bold;
                    color:#b45309;
                    margin-bottom:12px;
                ">
                    ⚠ {titulo}
                </div>

                <div style="
                    font-size:21px;
                    font-weight:bold;
                    color:#333333;
                    line-height:1.5;
                ">
                    {mensaje}
                </div>

            </div>
            """
        )
    )


# ================================================================
# CONFIGURACION GENERAL
# ================================================================

LIMITE_ANOMALIA = 2200

INTERVALO_MINUTOS = 20

INTERVALO_HORAS = (
    INTERVALO_MINUTOS / 60
)

SEPARACION_FILAS = 1


# ================================================================
# ESCALA TURBO PASTEL
# ================================================================

turbo_pastel = [

    [0.00, "#8F8BD8"],
    [0.10, "#7FA7E8"],
    [0.20, "#71C5E8"],
    [0.30, "#6EDDD0"],
    [0.40, "#91E6B0"],
    [0.50, "#C4ED8C"],
    [0.60, "#F1EE88"],
    [0.70, "#F7D487"],
    [0.80, "#F5B27F"],
    [0.90, "#F28D83"],
    [1.00, "#D97D82"]
]


# ================================================================
# ================================================================
#
# PASO 1
# PEDIR ARCHIVOS AL USUARIO
#
# ================================================================
# ================================================================

cartel_usuario(

    "PASO 1 — CARGAR ARCHIVOS",

    """
    Seleccioná uno o varios archivos
    <b>CSV o Excel</b> con los datos de caudal.<br><br>

    Las primeras tres columnas deben ser:<br>

    <b>FECHA &nbsp;&nbsp; | &nbsp;&nbsp; HORA
    &nbsp;&nbsp; | &nbsp;&nbsp; CAUDAL</b>
    """
)


uploaded = files.upload()


# ================================================================
# VERIFICAR ARCHIVOS
# ================================================================

if len(uploaded) == 0:

    raise ValueError(
        "No se cargó ningún archivo."
    )


# ================================================================
# LEER ARCHIVOS
# ================================================================

dfs = []


for nombre, contenido in uploaded.items():

    print(
        f"Leyendo: {nombre}"
    )

    if nombre.lower().endswith(".csv"):

        df_temp = pd.read_csv(
            io.BytesIO(contenido),
            header=None
        )

    elif nombre.lower().endswith(
        (".xlsx", ".xls")
    ):

        df_temp = pd.read_excel(
            io.BytesIO(contenido),
            header=None
        )

    else:

        print(
            f"Archivo ignorado: {nombre}"
        )

        continue

    dfs.append(
        df_temp
    )


# ================================================================
# VERIFICAR ARCHIVOS VALIDOS
# ================================================================

if len(dfs) == 0:

    raise ValueError(
        "No se encontró ningún CSV o Excel válido."
    )


# ================================================================
# UNIFICAR ARCHIVOS
# ================================================================

df = pd.concat(
    dfs,
    ignore_index=True
)


# ================================================================
# TOMAR PRIMERAS 3 COLUMNAS
# ================================================================

df = df.iloc[:, :3].copy()


df.columns = [

    "Fecha",

    "Hora",

    "Caudal"
]


# ================================================================
# CONVERTIR CAUDAL
# ================================================================

df["Caudal"] = pd.to_numeric(
    df["Caudal"],
    errors="coerce"
)


# ================================================================
# CREAR FECHA-HORA
# ================================================================

df["FechaHora"] = pd.to_datetime(

    df["Fecha"].astype(str)

    + " "

    + df["Hora"].astype(str),

    errors="coerce",

    dayfirst=True
)


# ================================================================
# ELIMINAR DATOS INVALIDOS
# ================================================================

df = df.dropna(

    subset=[
        "FechaHora",
        "Caudal"
    ]
)


if len(df) == 0:

    raise ValueError(
        "No quedaron datos válidos después de convertir Fecha, Hora y Caudal."
    )


# ================================================================
# ORDENAR
# ================================================================

df = (
    df
    .sort_values("FechaHora")
    .set_index("FechaHora")
)


# ================================================================
# REMUESTREO CADA 20 MINUTOS
# ================================================================

df20 = (

    df["Caudal"]

    .resample(
        f"{INTERVALO_MINUTOS}min"
    )

    .mean()

    .reset_index()
)


# ================================================================
# CREAR DIA
# ================================================================

df20["Dia"] = (

    df20["FechaHora"]

    .dt.strftime(
        "%Y-%m-%d"
    )
)


# ================================================================
# CREAR HORA
# ================================================================

df20["Hora"] = (

    df20["FechaHora"]

    .dt.strftime(
        "%H:%M"
    )
)


# ================================================================
# MATRIZ DIA-HORA
# ================================================================

matriz = pd.pivot_table(

    df20,

    index="Dia",

    columns="Hora",

    values="Caudal",

    aggfunc="mean"
)


# ================================================================
# ORDENAR HORAS
# ================================================================

matriz = matriz.reindex(

    sorted(
        matriz.columns
    ),

    axis=1
)


# ================================================================
# ASEGURAR NUMERICOS
# ================================================================

matriz_num = matriz.apply(

    pd.to_numeric,

    errors="coerce"
)


# ================================================================
# MOSTRAR MATRIZ
# ================================================================

display(
    HTML(
        """
        <div style="
            background:#e8f4fd;
            border:2px solid #3b82f6;
            border-radius:10px;
            padding:15px;
            margin-top:20px;
            font-size:20px;
            font-weight:bold;
            text-align:center;
        ">
            MATRIZ GENERADA CORRECTAMENTE
        </div>
        """
    )
)


print()

print(
    f"Cantidad de días: "
    f"{matriz.shape[0]}"
)

print(
    f"Cantidad de horarios: "
    f"{matriz.shape[1]}"
)

print()

display(
    matriz
)


# ================================================================
# DETECTAR MES EVALUADO
# ================================================================

fechas_convertidas = pd.to_datetime(

    matriz_num.index,

    errors="coerce"
)


fechas_validas = (

    fechas_convertidas[
        ~pd.isna(
            fechas_convertidas
        )
    ]
)


MESES_ES = {

    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}


if len(fechas_validas) > 0:

    meses_en_datos = sorted(

        set(

            zip(

                fechas_validas.year,

                fechas_validas.month
            )
        )
    )


    if len(meses_en_datos) == 1:

        anio_evaluado = (
            meses_en_datos[0][0]
        )

        numero_mes = (
            meses_en_datos[0][1]
        )

        nombre_mes = (
            MESES_ES[
                numero_mes
            ]
        )

        MES_EVALUADO = (

            f"{nombre_mes} "
            f"{anio_evaluado}"
        )

    else:

        fecha_inicio = (
            fechas_validas.min()
        )

        fecha_fin = (
            fechas_validas.max()
        )

        MES_EVALUADO = (

            f"{fecha_inicio.strftime('%d/%m/%Y')}"

            " al "

            f"{fecha_fin.strftime('%d/%m/%Y')}"
        )

else:

    MES_EVALUADO = (
        "Período no identificado"
    )


# ================================================================
# ================================================================
#
# PASO 2
# PEDIR NOMBRE AL USUARIO
#
# ================================================================
# ================================================================

cartel_usuario(

    "PASO 2 — ESCRIBIR NOMBRE DEL INFORME",

    f"""
    Mes evaluado:
    <b>{MES_EVALUADO}</b><br><br>

    Escribí abajo el nombre que querés usar
    para los <b>DOS ARCHIVOS</b>.<br><br>

    Se generarán automáticamente:<br>

    <b>Nombre.xlsx</b><br>
    <b>Nombre.html</b><br><br>

    No hace falta escribir la extensión.
    """
)


nombre_base = input(
    "\n>>> ESCRIBÍ AQUÍ EL NOMBRE DEL INFORME: "
).strip()


# ================================================================
# NOMBRE AUTOMATICO SI ESTA VACIO
# ================================================================

if nombre_base == "":

    nombre_base = (

        f"Caudal Agua "
        f"{MES_EVALUADO}"
    )


# ================================================================
# QUITAR EXTENSION SI FUE ESCRITA
# ================================================================

if nombre_base.lower().endswith(
    ".xlsx"
):

    nombre_base = (
        nombre_base[:-5]
    )


if nombre_base.lower().endswith(
    ".html"
):

    nombre_base = (
        nombre_base[:-5]
    )


# ================================================================
# LIMPIAR CARACTERES INVALIDOS
# ================================================================

nombre_base = re.sub(

    r'[<>:"/\\|?*]',

    "_",

    nombre_base
)


# ================================================================
# NOMBRES FINALES
# ================================================================

archivo_excel = (
    nombre_base
    + ".xlsx"
)


archivo_html = (
    nombre_base
    + ".html"
)


# ================================================================
# CONFIRMACION VISUAL
# ================================================================

display(
    HTML(
        f"""
        <div style="
            background:#dcfce7;
            border:3px solid #16a34a;
            border-radius:14px;
            padding:20px;
            margin:20px 0;
            text-align:center;
        ">

            <div style="
                font-size:24px;
                font-weight:bold;
                color:#166534;
            ">
                NOMBRE ACEPTADO
            </div>

            <br>

            <div style="
                font-size:18px;
            ">

                Excel:
                <b>{archivo_excel}</b>

                <br><br>

                HTML:
                <b>{archivo_html}</b>

            </div>

        </div>
        """
    )
)


# ================================================================
# ================================================================
#
# CREAR EXCEL
#
# ================================================================
# ================================================================

matriz.to_excel(

    archivo_excel,

    sheet_name="Matriz"
)


# ================================================================
# ABRIR WORKBOOK
# ================================================================

wb = load_workbook(
    archivo_excel
)


ws = wb["Matriz"]


# ================================================================
# FORMATO NUMERICO
# ================================================================

for fila in ws.iter_rows(

    min_row=2,

    min_col=2
):

    for celda in fila:

        if celda.value is not None:

            celda.number_format = (
                "0.00"
            )


# ================================================================
# CENTRAR HOJA
# ================================================================

for fila in ws.iter_rows():

    for celda in fila:

        celda.alignment = Alignment(

            horizontal="center",

            vertical="center"
        )


# ================================================================
# AJUSTAR COLUMNAS
# ================================================================

for columna in ws.columns:

    largo_max = 0

    letra = (
        columna[0]
        .column_letter
    )


    for celda in columna:

        try:

            if celda.value is not None:

                largo_max = max(

                    largo_max,

                    len(
                        str(
                            celda.value
                        )
                    )
                )

        except:

            pass


    ws.column_dimensions[
        letra
    ].width = min(

        largo_max + 2,

        18
    )


# ================================================================
# CONGELAR ENCABEZADOS
# ================================================================

ws.freeze_panes = "B2"


# ================================================================
# AUTOFILTRO
# ================================================================

ws.auto_filter.ref = (
    ws.dimensions
)


# ================================================================
# GUARDAR EXCEL
# ================================================================

wb.save(
    archivo_excel
)


# ================================================================
# ETIQUETAS DE FECHA
# ================================================================

fechas_texto = (

    matriz_num.index

    .astype(str)

    .tolist()
)


# ================================================================
# FUNCION DURACION MAXIMA BAJO LIMITE
# ================================================================

def duracion_maxima_bajo_limite(
    fila,
    limite,
    intervalo_horas
):

    max_consecutivos = 0

    consecutivos = 0


    for valor in fila:

        if (
            pd.notna(valor)
            and valor < limite
        ):

            consecutivos += 1

            max_consecutivos = max(

                max_consecutivos,

                consecutivos
            )

        else:

            consecutivos = 0


    return (

        max_consecutivos

        * intervalo_horas
    )


# ================================================================
# ESTADISTICAS DIARIAS
# ================================================================

estadisticas = pd.DataFrame(

    index=matriz_num.index
)


estadisticas["Minimo"] = (

    matriz_num.min(
        axis=1
    )
)


estadisticas["Promedio"] = (

    matriz_num.mean(
        axis=1
    )
)


estadisticas["Maximo"] = (

    matriz_num.max(
        axis=1
    )
)


estadisticas["Rango"] = (

    estadisticas["Maximo"]

    -

    estadisticas["Minimo"]
)


estadisticas[
    "Duracion_Bajo_Limite"
] = matriz_num.apply(

    lambda fila:

    duracion_maxima_bajo_limite(

        fila,

        LIMITE_ANOMALIA,

        INTERVALO_HORAS
    ),

    axis=1
)


# ================================================================
# INDICADORES GENERALES
# ================================================================

minimo_general = (

    matriz_num
    .min()
    .min()
)


promedio_general = (

    matriz_num
    .stack()
    .mean()
)


maximo_general = (

    matriz_num
    .max()
    .max()
)


max_evento_general = (

    estadisticas[
        "Duracion_Bajo_Limite"
    ]
    .max()
)


# ================================================================
# BUSCAR ANOMALIAS
# ================================================================

anom_x = []

anom_y = []


for fila, dia in enumerate(
    matriz_num.index
):

    for col, hora in enumerate(
        matriz_num.columns
    ):

        valor = matriz_num.iloc[
            fila,
            col
        ]


        if pd.notna(valor):

            if valor < LIMITE_ANOMALIA:

                anom_x.append(
                    str(hora)
                )

                anom_y.append(
                    str(dia)
                )


# ================================================================
# ================================================================
#
# HEATMAP
#
# ================================================================
# ================================================================

fig_heatmap = go.Figure()


fig_heatmap.add_trace(

    go.Heatmap(

        z=matriz_num.values,

        x=matriz_num.columns.astype(
            str
        ),

        y=matriz_num.index.astype(
            str
        ),

        colorscale=turbo_pastel,

        colorbar=dict(

            title="Caudal (m³/h)",

            thickness=18,

            len=0.90
        ),

        xgap=0,

        ygap=SEPARACION_FILAS,

        hovertemplate=

        "<b>Fecha:</b> %{y}"

        "<br><b>Hora:</b> %{x}"

        "<br><b>Caudal:</b> "
        "%{z:.2f} m³/h"

        "<extra></extra>"
    )
)


# ================================================================
# ANOMALIAS
# ================================================================

fig_heatmap.add_trace(

    go.Scatter(

        x=anom_x,

        y=anom_y,

        mode="markers",

        name=(

            f"Anomalías < "
            f"{LIMITE_ANOMALIA} m³/h"
        ),

        marker=dict(

            symbol="square-open",

            size=10,

            color="red",

            line=dict(

                color="red",

                width=2
            )
        ),

        hovertemplate=

        "⚠ <b>ANOMALÍA</b>"

        "<br>Fecha: %{y}"

        "<br>Hora: %{x}"

        "<extra></extra>"
    )
)


# ================================================================
# ALTURA HEATMAP
# ================================================================

alto_heatmap = max(

    400,

    len(
        matriz_num.index
    ) * 15
)


# ================================================================
# LAYOUT HEATMAP
# ================================================================

fig_heatmap.update_layout(

    title=dict(

        text=(

            "Heatmap Operativo de Caudal"

            "<br>"

            f"<sup>{MES_EVALUADO}</sup>"
        ),

        x=0.5,

        xanchor="center",

        font=dict(
            size=22
        )
    ),

    xaxis_title="Hora",

    yaxis_title="Día",

    height=alto_heatmap,

    template="plotly_white",

    plot_bgcolor="white",

    paper_bgcolor="white",

    margin=dict(

        l=120,

        r=30,

        t=100,

        b=70
    ),

    legend=dict(

        orientation="h",

        y=1.05,

        x=1,

        xanchor="right"
    )
)


fig_heatmap.update_xaxes(

    tickangle=90,

    tickfont=dict(
        size=7
    ),

    showgrid=False
)


fig_heatmap.update_yaxes(

    tickmode="array",

    tickvals=(
        matriz_num.index.astype(str)
    ),

    ticktext=(
        matriz_num.index.astype(str)
    ),

    tickfont=dict(
        size=8
    ),

    showgrid=False
)


# ================================================================
# ================================================================
#
# COMPORTAMIENTO DIARIO
#
# ================================================================
# ================================================================

fig_estadisticas = go.Figure()


# MAXIMO

fig_estadisticas.add_trace(

    go.Scatter(

        x=fechas_texto,

        y=estadisticas["Maximo"],

        mode="lines",

        name="Máximo diario",

        line=dict(

            color="rgba(100,130,160,0.65)",

            width=1
        ),

        hoverinfo="skip"
    )
)


# MINIMO + BANDA

fig_estadisticas.add_trace(

    go.Scatter(

        x=fechas_texto,

        y=estadisticas["Minimo"],

        mode="lines",

        name="Rango Mínimo - Máximo",

        line=dict(

            color="rgba(100,130,160,0.65)",

            width=1
        ),

        fill="tonexty",

        fillcolor=(
            "rgba(150,190,220,0.28)"
        ),

        hoverinfo="skip"
    )
)


# ================================================================
# DATOS PARA HOVER
# ================================================================

custom_hover = np.column_stack((

    estadisticas["Minimo"],

    estadisticas["Maximo"],

    estadisticas["Rango"]
))


# ================================================================
# PROMEDIO
# ================================================================

fig_estadisticas.add_trace(

    go.Scatter(

        x=fechas_texto,

        y=estadisticas["Promedio"],

        mode="lines+markers",

        name="Promedio diario",

        line=dict(

            color="#315F85",

            width=3
        ),

        marker=dict(

            size=6,

            color="#315F85"
        ),

        customdata=custom_hover,

        hovertemplate=

        "<b>Fecha:</b> %{x}"

        "<br><br>"

        "<b>Mínimo:</b> "
        "%{customdata[0]:,.0f} m³/h"

        "<br><b>Promedio:</b> "
        "%{y:,.0f} m³/h"

        "<br><b>Máximo:</b> "
        "%{customdata[1]:,.0f} m³/h"

        "<br><br>"

        "<b>Rango diario:</b> "
        "%{customdata[2]:,.0f} m³/h"

        "<extra></extra>"
    )
)


# ================================================================
# LINEA LIMITE
# ================================================================

fig_estadisticas.add_hline(

    y=LIMITE_ANOMALIA,

    line_dash="dash",

    line_color="red",

    line_width=2,

    annotation_text=(

        f"Límite "
        f"{LIMITE_ANOMALIA} m³/h"
    ),

    annotation_position="bottom left"
)


# ================================================================
# CONTROL FECHAS
# ================================================================

cantidad_dias = len(
    fechas_texto
)


if cantidad_dias <= 15:

    salto_fecha = 1

elif cantidad_dias <= 35:

    salto_fecha = 2

elif cantidad_dias <= 70:

    salto_fecha = 5

elif cantidad_dias <= 150:

    salto_fecha = 10

else:

    salto_fecha = 15


tickvals_estadisticas = (

    fechas_texto[
        ::salto_fecha
    ]
)


# ================================================================
# LAYOUT ESTADISTICAS
# ================================================================

fig_estadisticas.update_layout(

    title=dict(

        text=(

            "Comportamiento Diario del Caudal"

            "<br>"

            f"<sup>{MES_EVALUADO} | "
            "Banda = mínimo-máximo | "
            "Línea = promedio diario</sup>"
        ),

        x=0.5,

        xanchor="center"
    ),

    xaxis_title="Día",

    yaxis_title="Caudal (m³/h)",

    height=520,

    template="plotly_white",

    hovermode="closest",

    margin=dict(

        l=90,

        r=40,

        t=110,

        b=90
    ),

    legend=dict(

        orientation="h",

        y=1.08,

        x=0.5,

        xanchor="center"
    )
)


fig_estadisticas.update_xaxes(

    tickmode="array",

    tickvals=(
        tickvals_estadisticas
    ),

    ticktext=(
        tickvals_estadisticas
    ),

    tickangle=45,

    tickfont=dict(
        size=9
    ),

    showgrid=False
)


fig_estadisticas.update_yaxes(

    showgrid=True,

    gridcolor=(
        "rgba(200,200,200,0.35)"
    ),

    zeroline=False
)


# ================================================================
# ================================================================
#
# DURACION BAJO LIMITE
#
# ================================================================
# ================================================================

fig_duracion = go.Figure()


fig_duracion.add_trace(

    go.Bar(

        x=fechas_texto,

        y=estadisticas[
            "Duracion_Bajo_Limite"
        ],

        name="Duración máxima",

        marker=dict(

            color=estadisticas[
                "Duracion_Bajo_Limite"
            ],

            colorscale=[

                [0.00, "#F9E3E3"],

                [0.50, "#F3AAAA"],

                [1.00, "#D95F5F"]
            ]
        ),

        hovertemplate=

        "<b>Fecha:</b> %{x}"

        "<br><br>"

        "<b>Máx. continuo "
        f"< {LIMITE_ANOMALIA}:</b> "

        "%{y:.2f} horas"

        "<extra></extra>"
    )
)


fig_duracion.update_layout(

    title=dict(

        text=(

            "Duración Máxima Continua "
            "de Bajo Caudal"

            "<br>"

            f"<sup>{MES_EVALUADO} | "
            f"Por debajo de "
            f"{LIMITE_ANOMALIA} m³/h</sup>"
        ),

        x=0.5,

        xanchor="center"
    ),

    xaxis_title="Día",

    yaxis_title="Duración (horas)",

    height=450,

    template="plotly_white",

    margin=dict(

        l=90,

        r=40,

        t=110,

        b=90
    ),

    showlegend=False
)


fig_duracion.update_xaxes(

    tickmode="array",

    tickvals=(
        tickvals_estadisticas
    ),

    ticktext=(
        tickvals_estadisticas
    ),

    tickangle=45,

    tickfont=dict(
        size=9
    ),

    showgrid=False
)


fig_duracion.update_yaxes(

    showgrid=True,

    gridcolor=(
        "rgba(200,200,200,0.35)"
    ),

    zeroline=False
)


# ================================================================
# MOSTRAR GRAFICOS
# ================================================================

fig_heatmap.show()

fig_estadisticas.show()

fig_duracion.show()


# ================================================================
# TABLA RESUMEN
# ================================================================

tabla = estadisticas.copy()


tabla.columns = [

    "Mínimo (m³/h)",

    "Promedio (m³/h)",

    "Máximo (m³/h)",

    "Rango (m³/h)",

    (
        f"Máx. continuo < "
        f"{LIMITE_ANOMALIA} (h)"
    )
]


tabla = tabla.round(2)


tabla_html = tabla.to_html(

    classes="tabla-resumen",

    border=0
)


# ================================================================
# CONVERTIR GRAFICOS A HTML
# ================================================================

html_heatmap = plot(

    fig_heatmap,

    include_plotlyjs=True,

    output_type="div",

    config={
        "responsive": True
    }
)


html_estadisticas = plot(

    fig_estadisticas,

    include_plotlyjs=False,

    output_type="div",

    config={
        "responsive": True
    }
)


html_duracion = plot(

    fig_duracion,

    include_plotlyjs=False,

    output_type="div",

    config={
        "responsive": True
    }
)


# ================================================================
# HTML FINAL
# ================================================================

html_final = f"""

<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta
name="viewport"
content="width=device-width, initial-scale=1.0"
>

<title>
Análisis Operativo de Caudal - {MES_EVALUADO}
</title>


<style>

body {{

    font-family: Arial, Helvetica, sans-serif;

    background-color: #ffffff;

    margin: 30px;

    color: #333333;
}}


h1 {{

    text-align: center;

    margin-bottom: 5px;
}}


.mes-evaluado {{

    text-align: center;

    font-size: 24px;

    font-weight: bold;

    color: #315F85;

    margin-top: 10px;

    margin-bottom: 10px;
}}


.descripcion {{

    text-align: center;

    color: #666666;

    margin-bottom: 25px;
}}


h2 {{

    text-align: center;

    margin-top: 50px;

    margin-bottom: 10px;
}}


.seccion {{

    margin-bottom: 60px;

    width: 100%;
}}


.indicadores {{

    display: flex;

    flex-wrap: wrap;

    justify-content: center;

    gap: 20px;

    margin-top: 30px;

    margin-bottom: 40px;
}}


.tarjeta {{

    background-color: #f7f8fa;

    border: 1px solid #dddddd;

    border-radius: 10px;

    padding: 18px 25px;

    min-width: 190px;

    text-align: center;

    box-shadow:
    0 2px 5px rgba(0,0,0,0.06);
}}


.tarjeta-titulo {{

    font-size: 13px;

    color: #666666;

    margin-bottom: 8px;
}}


.tarjeta-valor {{

    font-size: 22px;

    font-weight: bold;

    color: #333333;
}}


.tabla-contenedor {{

    width: 95%;

    margin-left: auto;

    margin-right: auto;

    overflow-x: auto;
}}


.tabla-resumen {{

    border-collapse: collapse;

    width: 100%;

    margin-top: 30px;

    font-size: 13px;
}}


.tabla-resumen th {{

    background-color: #eeeeee;

    padding: 9px;

    border-bottom: 2px solid #cccccc;

    position: sticky;

    top: 0;
}}


.tabla-resumen td {{

    padding: 8px;

    text-align: center;

    border-bottom: 1px solid #dddddd;
}}


.tabla-resumen tr:hover {{

    background-color: #f5f5f5;
}}

</style>

</head>


<body>


<h1>
Análisis Operativo de Caudal
</h1>


<div class="mes-evaluado">

Mes evaluado:
{MES_EVALUADO}

</div>


<div class="descripcion">

Límite de anomalía:

<b>
{LIMITE_ANOMALIA:,.0f} m³/h
</b>

</div>


<div class="indicadores">


<div class="tarjeta">

<div class="tarjeta-titulo">
Mínimo del mes
</div>

<div class="tarjeta-valor">
{minimo_general:,.0f} m³/h
</div>

</div>


<div class="tarjeta">

<div class="tarjeta-titulo">
Promedio del mes
</div>

<div class="tarjeta-valor">
{promedio_general:,.0f} m³/h
</div>

</div>


<div class="tarjeta">

<div class="tarjeta-titulo">
Máximo del mes
</div>

<div class="tarjeta-valor">
{maximo_general:,.0f} m³/h
</div>

</div>


<div class="tarjeta">

<div class="tarjeta-titulo">
Mayor evento continuo &lt; {LIMITE_ANOMALIA}
</div>

<div class="tarjeta-valor">
{max_evento_general:.2f} h
</div>

</div>


</div>


<div class="seccion">

{html_heatmap}

</div>


<div class="seccion">

<h2>
Comportamiento Diario
</h2>

{html_estadisticas}

</div>


<div class="seccion">

<h2>
Eventos de Bajo Caudal
</h2>

{html_duracion}

</div>


<div class="seccion">

<h2>
Resumen Diario
</h2>


<div class="tabla-contenedor">

{tabla_html}

</div>


</div>


</body>

</html>

"""


# ================================================================
# GUARDAR HTML
# ================================================================

with open(

    archivo_html,

    "w",

    encoding="utf-8"

) as archivo:

    archivo.write(
        html_final
    )


# ================================================================
# CARTEL FINAL
# ================================================================

display(
    HTML(
        f"""
        <div style="
            background:#dcfce7;
            border:4px solid #16a34a;
            border-radius:16px;
            padding:25px;
            margin:30px 0;
            text-align:center;
            box-shadow:0 4px 12px rgba(0,0,0,0.15);
        ">

            <div style="
                font-size:30px;
                font-weight:bold;
                color:#166534;
            ">
                ✓ ANÁLISIS COMPLETADO
            </div>

            <br>

            <div style="
                font-size:20px;
                line-height:1.7;
            ">

                Mes evaluado:
                <b>{MES_EVALUADO}</b>

                <br>

                Excel:
                <b>{archivo_excel}</b>

                <br>

                HTML:
                <b>{archivo_html}</b>

            </div>

        </div>
        """
    )
)


# ================================================================
# DESCARGAR ARCHIVOS
# ================================================================

cartel_usuario(

    "PASO 3 — DESCARGA DE RESULTADOS",

    f"""
    El análisis terminó correctamente.<br><br>

    Ahora se descargarán los dos archivos:<br><br>

    📊 <b>{archivo_excel}</b><br><br>

    📈 <b>{archivo_html}</b>
    """
)


files.download(
    archivo_excel
)


files.download(
    archivo_html
)

Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL (4).csv
Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1 (4).csv
Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2 (4).csv
Leyendo: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL (4).csv
Leyendo: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1 (4).csv
Leyendo: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2 (4).csv


/tmp/ipykernel_714/3230739073.py:251: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.




Cantidad de días: 6
Cantidad de horarios: 72



Hora,00:00,00:20,00:40,01:00,01:20,01:40,02:00,02:20,02:40,03:00,...,20:40,21:00,21:20,21:40,22:00,22:20,22:40,23:00,23:20,23:40
Dia,,,,,,,,,,,,,,,,,,,,,
2026-10-01,6469.560554,6459.945824,6495.775281,6450.966368,6434.427602,6550.963964,6441.375839,6349.141256,6490.194131,6507.789116,...,6865.049550,6952.569820,6852.977427,6929.630872,7079.526786,7274.038031,7333.319728,7026.298206,6817.194131,6939.306488
2026-10-02,7100.363229,7075.192308,6987.011236,6988.968610,6934.140625,7048.092971,6772.367946,6731.098876,6780.242697,6799.874157,...,7168.162162,7220.087640,7251.127854,7158.789593,7095.986517,7173.936652,7074.308390,7115.849658,6807.676404,6672.724832
2026-10-03,6868.599099,6977.619590,6865.970655,6721.691275,6876.731982,6849.561364,6960.605442,6949.912162,6769.024609,6773.011261,...,6589.482143,6784.831839,6983.654628,7075.049887,6937.368778,7198.467416,7173.082960,6855.431151,7059.432127,7354.051802
2026-10-04,7090.596413,6991.840449,6977.559361,6975.713004,6795.937079,6641.819820,6696.841629,6381.839729,6524.316027,6921.161798,...,5331.434685,5276.874439,5314.845638,5246.342404,5424.442922,5224.827273,5292.022422,5396.900677,5263.931507,5502.995455
2026-10-05,5464.521445,5664.286036,5837.750567,5675.239726,5721.191874,5596.355056,5699.130926,5509.726862,5651.817156,5485.221973,...,6302.130926,6128.922374,6155.223476,6358.405830,6280.244898,6233.609091,6223.599089,6218.265766,6319.373034,6259.275395
2026-10-06,6153.627273,6410.688488,6325.492135,6088.087444,5876.834091,6081.467120,6270.162162,6279.695067,6259.353604,6185.886105,...,6703.648649,6563.877551,6685.388128,6537.589165,6803.796840,6553.128959,6493.063781,6582.247166,6514.641084,6684.160271



>>> ESCRIBÍ AQUÍ EL NOMBRE DEL INFORME: CAUDAL AGUA B OCT26


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>